# 16 — Sintéticos II: validación de fidelidad y utilidad

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Decidir qué generadores de `15` son **admisibles** para el laboratorio (`17`) y el aumento de
datos (`18`), midiendo fidelidad (¿se parecen al real?), utilidad (¿sirven para entrenar?) y
privacidad/memorización (¿copian el entrenamiento?), siempre contra el **tramo de
entrenamiento** real con el que se ajustaron.

## Preguntas que responde

1. ¿Reproducen los hechos estilizados (colas pesadas, ausencia de autocorrelación en
   rentabilidades, agrupamiento de volatilidad, efecto apalancamiento)?
2. ¿Coinciden la distribución de duraciones de régimen y la matriz de transición con las reales?
3. ¿Se conserva la estructura de correlación **por régimen** (p. ej. el aumento de correlaciones
   en crisis)?
4. ¿Distingue un clasificador real de sintético mejor que el azar (score discriminativo)?
5. ¿Un detector entrenado en sintético funciona en real (TSTR)?
6. ¿Hay indicios de memorización?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| trayectorias sintéticas | `rutas.DATA_SINTETICOS` |
| paneles reales (solo tramo de entrenamiento) | `rutas.DATA_PROCESSED` |
| configuración | `rutas.SINTETICOS_CONFIG` |
| tablas de validación y figuras | `rutas.RESULTS_SINTETICOS / 'validacion'` |

## Metodología prevista

Funciones de `regimenes.sinteticos.validacion` (firmas ya fijadas):

- **Hechos estilizados** (Cont 2001; `hmm_bullabulla2006`) — `fidelidad_marginal` y
  `fidelidad_dependencia`: KS y momentos por feature, índice de cola, ACF de $r_t$, $|r_t|$ y
  $r_t^2$, correlación $\operatorname{corr}(r_t, \sigma_{t+1})$.
- **Duración y transiciones** — `fidelidad_regimenes`: histograma de duraciones por régimen y
  distancia entre matrices de transición real y sintética.
- **Correlaciones por régimen**: distancia (Frobenius) entre matrices de correlación real y
  sintética condicionadas a cada régimen.
- **Score discriminativo**: clasificador post-hoc (p. ej. GRU o gradient boosting sobre ventanas)
  real vs sintético con validación cruzada; se reporta $|\text{acc} - 0{,}5|$ (Yoon et al. 2019).
- **TSTR** — `utilidad_tstr` (Esteban, Hyland y Rätsch 2017): se ajusta un detector del registro
  sobre sintético y se evalúa sobre el real con las métricas del benchmark; se compara con TRTR
  (entrenar y evaluar en real) dentro del mismo tramo.
- **Memorización** — `memorizacion`: distancia de cada ventana sintética a su vecina real más
  cercana frente a la distancia real–real.

Criterio de admisión: umbrales fijados en la celda de configuración **antes** de ver los
resultados; el veredicto se imprime por celda.

Referencias pendientes de añadir a `docs/references.bib`: Cont (2001), Esteban et al. (2017),
Yoon et al. (2019).

## Riesgos

- **Fuga temporal.** Todo generador se ajusta **solo** con el tramo de entrenamiento
  (`entrenamiento.fin_train` de `configs/sinteticos.yaml`); normalizaciones, selección de
  hiperparámetros y etiquetas de régimen de referencia también se calculan solo con ese tramo.
  Mientras `fin_train` sea `null` no se puede pasar `IMPLEMENTADO=True` (la celda de
  configuración lo comprueba).
- **Circularidad.** Si las etiquetas de régimen con las que se condiciona el generador salen de un
  detector del benchmark, ese detector (y su familia) juega en casa al evaluarse sobre lo
  generado. Se mitiga con etiquetas de referencia independientes de los detectores (ventanas de
  crisis y suelos de drawdown de `configs/benchmark_spec.yaml`, o varias etiquetas de familias
  distintas) y leyendo la matriz generador × detector con la diagonal "misma familia" aparte.
- **Validar contra el OOS.** La validación usa solo el tramo de entrenamiento; comparar con el
  tramo OOS convertiría la elección del generador en una fuga hacia `17`–`18`.
- **Métricas que se compensan.** Un generador puede acertar marginales y fallar dependencias; el
  veredicto exige todas las dimensiones, no un promedio.

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Hechos estilizados
3. Duración y transiciones de régimen
4. Correlaciones por régimen
5. Score discriminativo
6. TSTR (entrenar en sintético, evaluar en real)
7. Memorización
8. Veredicto por generador


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')
from regimenes.sinteticos import validacion

RUTAS_NB = {
    'entrada: trayectorias sintéticas': rutas.DATA_SINTETICOS,
    'entrada: paneles reales': rutas.DATA_PROCESSED,
    'entrada: configuración sintéticos': rutas.SINTETICOS_CONFIG,
    'salida: validación': rutas.RESULTS_SINTETICOS / 'validacion',
}
display(tabla_rutas(RUTAS_NB))
print('Métricas declaradas en la configuración:', CONFIG_SINTETICOS['validacion']['metricas'])
print('Generadores registrados:', sorted(registro_sinteticos.GENERADORES) or 'ninguno todavía')
# Umbrales de admisión: se fijan aquí ANTES de ver resultados (pendiente).
UMBRALES_ADMISION = {}
print('Umbrales de admisión fijados:', UMBRALES_ADMISION or 'pendientes')


,ruta,existe
papel,,
entrada: trayectorias sintéticas,data/sinteticos,True
entrada: paneles reales,data/processed,True
entrada: configuración sintéticos,configs/sinteticos.yaml,True
salida: validación,results/sinteticos/validacion,False


Métricas declaradas en la configuración: ['fidelidad_marginal', 'fidelidad_dependencia', 'fidelidad_regimenes', 'utilidad_tstr', 'memorizacion']
Generadores registrados: ninguno todavía
Umbrales de admisión fijados: pendientes


In [2]:
fin_train = CONFIG_SINTETICOS['entrenamiento'].get('fin_train')
if fin_train is None:
    mensaje = ('configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de '
               'entrenamiento antes de ajustar cualquier generador (regla anti-fuga).')
    if IMPLEMENTADO:
        raise ValueError(mensaje)
    print('[aviso]', mensaje)
else:
    print('Corte de entrenamiento de los generadores:', fin_train)


[aviso] configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de entrenamiento antes de ajustar cualquier generador (regla anti-fuga).


## 2. Hechos estilizados

In [3]:
# TODO 16.2 — hechos estilizados
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. real = panel de train; sint = trayectorias de rutas.DATA_SINTETICOS por generador
    # 2. validacion.fidelidad_marginal(real, sint) y validacion.fidelidad_dependencia(real, sint)
    # 3. figuras: QQ de colas, ACF de r, |r| y r^2 real vs sintético
    raise NotImplementedError('TODO 16.2: implementar validacion.fidelidad_marginal / fidelidad_dependencia')
else:
    saltar('16.2', 'hechos estilizados')


[TODO pendiente] 16.2: hechos estilizados


## 3. Duración y transiciones de régimen

In [4]:
# TODO 16.3 — duración y transiciones de régimen
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. validacion.fidelidad_regimenes(regimen_ref_train, sint): duraciones y matriz de transición
    raise NotImplementedError('TODO 16.3: implementar validacion.fidelidad_regimenes')
else:
    saltar('16.3', 'duración y transiciones de régimen')


[TODO pendiente] 16.3: duración y transiciones de régimen


## 4. Correlaciones por régimen

In [5]:
# TODO 16.4 — correlaciones por régimen
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. matriz de correlación por régimen (real train vs sintético) y distancia de Frobenius
    raise NotImplementedError('TODO 16.4: comparar correlaciones condicionadas a régimen')
else:
    saltar('16.4', 'correlaciones por régimen')


[TODO pendiente] 16.4: correlaciones por régimen


## 5. Score discriminativo

In [6]:
# TODO 16.5 — score discriminativo
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. clasificador real vs sintético sobre ventanas, validación cruzada por bloques; |acc - 0,5|
    raise NotImplementedError('TODO 16.5: implementar el score discriminativo')
else:
    saltar('16.5', 'score discriminativo')


[TODO pendiente] 16.5: score discriminativo


## 6. TSTR (entrenar en sintético, evaluar en real)

In [7]:
# TODO 16.6 — TSTR frente a TRTR
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes.detectores import detector_specs

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. validacion.utilidad_tstr(real, sint, detector_id) para detectores del registro
    # 2. referencia TRTR en el mismo tramo real; sin tocar el OOS del benchmark
    raise NotImplementedError('TODO 16.6: implementar validacion.utilidad_tstr')
else:
    saltar('16.6', 'TSTR frente a TRTR')


[TODO pendiente] 16.6: TSTR frente a TRTR


## 7. Memorización

In [8]:
# TODO 16.7 — memorización
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. validacion.memorizacion(real, sint): distancia al vecino real más cercano vs real-real
    raise NotImplementedError('TODO 16.7: implementar validacion.memorizacion')
else:
    saltar('16.7', 'memorización')


[TODO pendiente] 16.7: memorización


## 8. Veredicto por generador

Tabla generador × dimensión con admitido/no admitido según `UMBRALES_ADMISION`, guardada en `rutas.RESULTS_SINTETICOS / 'validacion'`. Siguiente: `17_sinteticos_laboratorio`.

In [9]:
# TODO 16.8 — veredicto de admisión
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. combinar 16.2-16.7 con UMBRALES_ADMISION; guardar veredicto.csv
    raise NotImplementedError('TODO 16.8: construir el veredicto de admisión')
else:
    saltar('16.8', 'veredicto de admisión')


[TODO pendiente] 16.8: veredicto de admisión
